# Day 2-04：VLMに画像を見せる

画像と言語を入力に受け付けるモデル（Vision Language Model; VLM）をダウンロードします．

さまざまな画像と質問を入力し、答えが変わる条件を調べます。

## 実行前の準備

**`DRIVE_PROJECT`を自分のDrive内の教材フォルダに合わせてください。**
ローカルでは`notebooks/day2/`または教材ルートから実行します。ColabではGPUランタイムを選びます。

In [ ]:
#@title 環境設定（最初に実行）
from pathlib import Path
import runpy

###################################################################
# JKJ1A/へ移動．
DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/JKJ1A")  # !!! 自分の環境に合わせて変更
LOCAL_PROJECT = (
    Path.cwd()
    if (Path.cwd() / "jkj1a/launcher.py").is_file()
    else Path.cwd().parent.parent
)
###################################################################

try:
    from google.colab import drive
    ON_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    project_hint = DRIVE_PROJECT
except ImportError:
    ON_COLAB = False
    project_hint = LOCAL_PROJECT

_context = runpy.run_path(str(project_hint / "jkj1a/launcher.py"))
ON_COLAB = _context["ON_COLAB"]
project_root = _context["project_root"]
print("実行場所:", "Colab" if ON_COLAB else "ローカル")
print("作業フォルダ:", project_root)

## 1. 小型VLM

分類器が固定されたクラスを選ぶのに対し、VLMは画像と文章から回答を生成します。ここでは英語の質問を使います。モデルの読み込みは最初の1回だけ行います。

ここでは[SmolVLM-500M-Instruct](https://huggingface.co/HuggingFaceTB/SmolVLM-500M-Instruct)を使います。約5億パラメータの学習済みモデルです。

`MiniVLM`は、画像と質問の前処理や回答の生成をまとめた補助クラスです。`vlm.ask(image, question)`には画像1枚（PIL画像）と質問文（文字列）を渡し、回答文（文字列）を受け取ります。


In [ ]:
from jkj1a.vision_language import MiniVLM

# 画像と質問文を入力できる学習済みモデルを読み込む
vlm = MiniVLM(model_id="HuggingFaceTB/SmolVLM-500M-Instruct")

## 2. 鹿を認識する

通常の鹿の画像を見せ、動物の種類を正しく答えられるか確認します。画像を表示して、自分でも動物の種類と足の本数を確かめてください。

教材ルートの`sample-images/`に好きな画像（JPEG・PNGなど）を入れ、次のセルのファイル名を変更して試してください。Colabでは、Drive内の教材フォルダにある`sample-images/`へ画像を追加します。質問も画像の内容に合わせて変えてください。

画像は`vlm.ask`の内部で、モデル付属の`AutoProcessor`を使って前処理されます。モデルに合わせた画像のサイズ調整や正規化・テンソル化、質問文のトークン化が行われるので、ここで手動でリサイズや正規化をする必要はありません。

サンプル画像として`dog.jpg`（犬）、`cat.png`（猫）、`astronaut.png`（宇宙飛行士）、`rocket.jpg`（ロケット）を用意しています。人物なら服の色、ロケットなら写っている物体など、画像に合わせて質問を変えてみてください。


In [ ]:
from pathlib import Path
from PIL import Image, ImageOps

# sample-imagesに入れた画像のファイル名を指定する
image_path = Path("sample-images/vlmbias-deer-original.jpg")

# 写真の向きを補正し、RGB画像として読み込む
with Image.open(image_path) as image:
    photo = ImageOps.exif_transpose(image).convert("RGB")
display(photo)


### このVLMは画像を理解しているのか？

In [ ]:
question = "What animal is in this image?"

# 画像と質問文を渡し、モデルの回答を表示する
answer = vlm.ask(photo, question)
print("質問:", question)
print("回答:", answer)

In [ ]:
question = "How many legs does this animal have?"

# 画像と質問文を渡し、モデルの回答を表示する
answer = vlm.ask(photo, question)
print("質問:", question)
print("回答:", answer)

### このVLMは画像を「本当に」理解しているのか？

鹿の画像を編集して5本脚にした

In [ ]:
# 足を1本増やした画像を読み込み、見える足を確認する
with Image.open("sample-images/vlmbias-deer.png") as image:
    edited_photo = ImageOps.exif_transpose(image).convert("RGB")
display(edited_photo)

In [ ]:
question = "How many legs does this animal have?"

# 画像と質問文を渡し、モデルの回答を表示する
answer = vlm.ask(photo, question)
print("質問:", question)
print("回答:", answer)

## 4. 青い円を数える

青い円の個数をプログラムで決めます。まず、文字を追加していない画像への回答を確認します。回答の生成ではサンプリングを使いません。

In [ ]:
from jkj1a.vision_language import counting_image

n = 5
# 青い円をn個、格子状に並べた画像を作る
image = counting_image(n=n, layout="grid", seed=0)
question = "How many blue circles are visible? Answer with only the number."

display(image)
# 画像と質問をモデルに渡し、回答文を受け取る
answer = vlm.ask(image, question)
print("正解:", n, "回答:", answer)

## 5. 配置を変える

個数を保ち、円の位置をずらします。各画像に対する回答をそのまま記録してください。

In [ ]:
# 条件ごとの回答を記録する
observations = []

# 円を規則的に並べる場合と、位置をずらす場合を比較する
for layout in ("grid", "jitter"):

    for n in (3, 5, 8):
        picture = counting_image(n=n, layout=layout, seed=0)
        response = vlm.ask(picture, question)
        observations.append({"layout": layout, "expected": n, "answer": response})

        display(picture)
        print(observations[-1], flush=True)

## 6. 記録を保存する

画像・質問・正解・モデルの回答・実行条件を保存します。同じ`name`で保存すると、警告を表示して画像と記録を上書きします。前の結果を残したい場合は`name`を変えてください。

In [ ]:
from jkj1a.vision_language import save_observation

picture = counting_image(n=5, layout="jitter", seed=1)
response = vlm.ask(picture, question)

# 入力画像・質問・回答・正解をまとめて保存する
path = save_observation(vlm, picture, question, response,
                        name="count-five-jitter-seed1", expected=5,
                        condition={"layout": "jitter", "seed": 1, "n": 5})
print(path)

## 演習

上記のVLMが，数え上げに対してどのような振る舞いをするか，画像や質問文をさまざま変化させパターンを観察せよ
- 画像の準備にAIモデル・画像生成モデルを活用してよい
- 成功・失敗の傾向はあるだろうか？